# CRSP characteristics and managed portfolios

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Loading database into google colab notebook environment**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp /content/drive/My\ Drive/Tidy\ Finance\ ML\ dataset/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!kaggle datasets download hansi666/tidy-finance-ml

# Step 4: Unzip the dataset (if necessary)
!unzip tidy-finance-ml.zip


In [ ]:
import shutil

# Path to the SQLite database in Google Drive
source_path = "/content/drive/My Drive/Tidy Finance ML dataset/tidy_finance_ML.sqlite"

# Destination path in Colab's local storage
destination_path = "/content/tidy_finance_ML.sqlite"

# Copy the file
shutil.copy(source_path, destination_path)


In [ ]:
!pip install rpy2
%load_ext rpy2.ipython



In [ ]:
!pip install joblib


In [ ]:
%%R
library(DBI)
# Your R code to load the dataframe goes here
# Install Packages
install.packages(c("tidyverse", "RSQLite","dplyr"))

# Load Libraries
library(dplyr)
library(tidyverse)
library(RSQLite)



**loading all the dataset**

In [ ]:
%%R
library(DBI)

# Point to the location of the SQLite file on Google Colab
file_path <- "/content/tidy_finance_ML.sqlite"

# Connect to tidy_finance
tidy_finance_ML <- dbConnect(SQLite(), file_path, extended_types = TRUE)


# Fetch all column names from the table
all_columns <- colnames(tbl(tidy_finance_ML, "stock_characteristics_monthly"))

# Filter columns that start with characteristic_ and macro_
all_stock_ch <- all_columns[startsWith(all_columns, "characteristic_")]
#all_macro_ch <- all_columns[startsWith(all_columns, "macro_")]
all_macro_ch <- c('macro_dp','macro_ep','macro_tbl','macro_svar','macro_bm', 'macro_dfy', 'macro_tms', 'macro_ntis')
# Combine selected columns with other required columns
selected_columns <- c('permno', 'month', 'ret_excess','mktcap_lag', 'sic2', all_macro_ch, all_stock_ch)
#
# Select used data
stock_characteristics_monthly <- tbl(tidy_finance_ML, "stock_characteristics_monthly") %>%
  select(all_of(selected_columns)) %>%
  collect() %>%
  drop_na()

# Count the number of stock characteristics
num_stock_ch <- length(all_stock_ch)
cat("Number of stock characteristics:", num_stock_ch, "\n")

# List all stock characteristics
cat("List of stock characteristics:\n")
print(all_stock_ch)

# Count the number of macro predictors
num_macro_ch <- length(all_macro_ch)
cat("Number of macro predictors:", num_macro_ch, "\n")

# List all macro predictors
cat("List of macro predictors:\n")
print(all_macro_ch)

# Group by month and count unique permno's
permno_count_per_month <- stock_characteristics_monthly %>%
  group_by(month) %>%
  summarise(permno_count = n_distinct(permno))

# Calculate average permno's per month
average_permno_per_month <- mean(permno_count_per_month$permno_count)
cat("Average number of permno's per month:", average_permno_per_month, "\n")

any(is.na(stock_characteristics_monthly))


**Most recent date**

In [ ]:
%%R
# Filter the data for the specified date range
filtered_data <- stock_characteristics_monthly %>%
  filter(month >= as.Date("1960-01-01") & month <= Sys.Date())

# Check the filtered data
print(filtered_data)


In [ ]:
%%R
# Filter the data for the specified date range
filtered_data <- stock_characteristics_monthly %>%
  filter(month >= as.Date("1960-01-01") & month <= as.Date("2000-01-01"))

# Check the filtered data
print(filtered_data)


In [ ]:
%%R
earliest_date <- min(stock_characteristics_monthly$month)
cat("Earliest date in the dataset:", earliest_date, "\n")

latest_date <- max(stock_characteristics_monthly$month)
cat("Latest date in the dataset:", latest_date, "\n")

cat("Date range in the dataset: from", earliest_date, "to", latest_date, "\n")


In [ ]:
%%R
# Ensure the month column is in Date format
stock_characteristics_monthly <- stock_characteristics_monthly %>%
  mutate(month = as.Date(month))

# Find the earliest and latest dates in the dataset
earliest_date <- min(stock_characteristics_monthly$month, na.rm = TRUE)
latest_date <- max(stock_characteristics_monthly$month, na.rm = TRUE)

cat("Earliest date in the dataset:", earliest_date, "\n")
cat("Latest date in the dataset:", latest_date, "\n")
cat("Date range in the dataset: from", earliest_date, "to", latest_date, "\n")


**converting R table into pandas datarame and deleting the old R table**

In [ ]:
# Import necessary libraries
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
import pandas as pd

# Activate pandas2ri conversion between R and pandas dataframes
pandas2ri.activate()

# Get the dataframe from the R environment
stock_characteristics_monthly_r = robjects.r['stock_characteristics_monthly']

# Use the conversion context to convert the R dataframe to a Pandas dataframe
with pandas2ri.conversion.localconverter(robjects.default_converter + pandas2ri.converter):
    stock_characteristics_monthly_py = pandas2ri.conversion.rpy2py(stock_characteristics_monthly_r)

# Now that the data is in a Pandas dataframe, delete the R object
del stock_characteristics_monthly_r

# Convert the numeric format of the month column to datetime format
stock_characteristics_monthly_py['month'] = pd.to_datetime(stock_characteristics_monthly_py['month'], origin='1970-01-01', unit='D')

# Display the first few rows of the modified DataFrame
print(stock_characteristics_monthly_py.head())
print(stock_characteristics_monthly_py.tail())

# Continue with the rest of your code...
# Here you can add additional data processing or analysis steps as needed


In [ ]:
# Convert the numeric format of the month column to datetime format
stock_characteristics_monthly_py['month'] = pd.to_datetime(stock_characteristics_monthly_py['month'])

# Find the date range
date_range = stock_characteristics_monthly_py['month'].agg(['min', 'max'])

# Display the date range
print(f"The dataset covers the period from {date_range['min']} to {date_range['max']}")


In [ ]:
!pip install pandas pyarrow
import pandas as pd

# Assuming you have your dataframe in the variable 'stock_characteristics_monthly_py'
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Save the dataframe as a Parquet file
stock_characteristics_monthly_py.to_parquet(file_path, engine='pyarrow')


**Load the Dataframe**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow

import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


In [ ]:
import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


In [ ]:
# Group by the 'month' column and count unique 'permno' values for each month
unique_permno_per_month = df.groupby('month')['permno'].nunique()

# Calculate the average number of unique 'permno' per month
average_unique_permno_per_month = unique_permno_per_month.mean()

# Print the result
print(f"The average number of unique permno per month is: {average_unique_permno_per_month}")


In [ ]:
# Find the earliest date in the 'month' column
earliest_date = stock_characteristics_monthly_py['month'].min()

# Print the earliest date
print("Earliest date in the dataset:", earliest_date)

# Find the earliest date in the 'month' column
latest_date = stock_characteristics_monthly_py['month'].max()

# Print the earliest date
print("latest date in the dataset:", latest_date)



In [ ]:
stock_characteristics_monthly_py.set_index(['month', 'permno'], inplace=True)
narratives = stock_characteristics_monthly_py.drop('ret_excess', axis=1).columns.tolist()
print(stock_characteristics_monthly_py.index)

In [ ]:
import pandas as pd

# Ensure the 'month' level of the index is a DatetimeIndex
stock_characteristics_monthly_py.index = stock_characteristics_monthly_py.index.set_levels([pd.to_datetime(stock_characteristics_monthly_py.index.levels[0]), stock_characteristics_monthly_py.index.levels[1]])

# Sort the MultiIndex to ensure it is lexsorted
stock_characteristics_monthly_py = stock_characteristics_monthly_py.sort_index()

# Define the date range
start_date = '1998-01-01'
end_date = '2000-12-31'

# Filter the DataFrame for the specified date range
filtered_df = stock_characteristics_monthly_py.loc[(slice(start_date, end_date), slice(None)), :]

# Display the filtered DataFrame
print(filtered_df.index)


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform

# Ensure the 'month' level of the index is a DatetimeIndex
stock_characteristics_monthly_py.index = stock_characteristics_monthly_py.index.set_levels([pd.to_datetime(stock_characteristics_monthly_py.index.levels[0]), stock_characteristics_monthly_py.index.levels[1]])

# Sort the MultiIndex to ensure it is lexsorted
stock_characteristics_monthly_py = stock_characteristics_monthly_py.sort_index()

# Define the date range
start_date = '1999-01-01'
end_date = '2000-12-31'

# Filter the DataFrame for the specified date range
filtered_df = stock_characteristics_monthly_py.loc[(slice(start_date, end_date), slice(None)), :]

# Extract the narratives column names
narratives = stock_characteristics_monthly_py.drop('ret_excess', axis=1).columns.tolist()

# Precompute quantile transformations for each month
def quantile_transform_group(group):
    transformed_data = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed_data, columns=group.columns, index=group.index)
    return transformed_df

# Apply quantile transformation group by group and scale the results
transformed_narratives = filtered_df.loc[:, narratives].groupby(level='month').apply(quantile_transform_group).mul(2).sub(1)
filtered_df.loc[:, narratives] = transformed_narratives

# Ensure ret_excess values are clipped between -1 and 1
filtered_df.loc[:, ['ret_excess']] = filtered_df.loc[:, ['ret_excess']].clip(lower=-1, upper=1.0)

# Display the filtered DataFrame
filtered_df.sort_index(inplace=True)
print(filtered_df)


**works, but is very slow, and macro_ are all equal to -1, because they are all the same each month**

In [ ]:
# Filter the list to exclude names starting with "macro"
filtered_narratives = [name for name in narratives if not name.startswith("macro")]

# Print the length of the filtered list
print(len(filtered_narratives))

In [ ]:
from sklearn.preprocessing import quantile_transform
from joblib import delayed, Parallel
from itertools import product
# Set the index of 'results' to a MultiIndex with 'date' and 'permno'
#merged_df.set_index(['date', 'permno'], inplace=True)
narratives = stock_characteristics_monthly_py.drop('ret_excess', axis=1).columns.tolist()

# narratives = results.columns.tolist()

# narratives = data.drop('returns', axis=1).columns.tolist()

filtered_df.loc[:, narratives] = (filtered_df.loc[:, narratives]
                                .groupby(level='month')
                                .apply(lambda x: pd.DataFrame(quantile_transform(x,
                                                                                 copy=True,
                                                                                 n_quantiles=x.shape[0]),
                                                              columns=narratives,
                                                              index=x.index.get_level_values('permno')))
                               .mul(2).sub(1))

filtered_df.sort_index(inplace=True)
filtered_df.index.names
filtered_df.loc[:, ['ret_excess']] = filtered_df.loc[:, ['ret_excess']].clip(lower=-1, upper=1.0)


In [ ]:
print(filtered_df)

In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform
from joblib import delayed, Parallel

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grCAPX", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Function to carry forward annual and quarterly data within a MultiIndex
def carry_forward(data, frequency, index_name):
    if frequency == 'annual':
        period = 12
    elif frequency == 'quarterly':
        period = 3
    else:
        raise ValueError("Frequency should be 'annual' or 'quarterly'")

    carried_data = data.copy()
    for i in range(1, period):
        carried_data = carried_data.groupby(level=index_name).ffill()
    return carried_data

# Apply carry forward for annual and quarterly data
for code in annual_codes:
    char_column = f'characteristic_{code}'
    stock_characteristics_monthly_py[char_column] = carry_forward(stock_characteristics_monthly_py[char_column], 'annual', 'permno')

for code in quarterly_codes:
    char_column = f'characteristic_{code}'
    stock_characteristics_monthly_py[char_column] = carry_forward(stock_characteristics_monthly_py[char_column], 'quarterly', 'permno')

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
narratives = [f'characteristic_{code}' for code in all_codes]

# Rank normalization function
def rank_normalize(df, columns):
    df[columns] = df[columns].apply(lambda x: pd.DataFrame(quantile_transform(x.values.reshape(-1, 1),
                                                                              copy=True,
                                                                              n_quantiles=len(x)).flatten(),
                                                           columns=[x.name],
                                                           index=x.index))
    df[columns] = df[columns] * 2 - 1
    return df

# Apply rank normalization by month
stock_characteristics_monthly_py = stock_characteristics_monthly_py.groupby('month').apply(rank_normalize, columns=narratives)

# Example of checking the result
print(stock_characteristics_monthly_py.head())


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform
from joblib import delayed, Parallel

# Assuming the data is loaded into 'stock_characteristics_df' with 'date' and 'permno' as columns
# Load your dataset
# stock_characteristics_df = pd.read_csv('your_data.csv')  # Example, replace with actual data loading

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grCAPX", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Function to carry forward annual and quarterly data
def carry_forward(data, frequency):
    if frequency == 'annual':
        period = 12
    elif frequency == 'quarterly':
        period = 3
    else:
        raise ValueError("Frequency should be 'annual' or 'quarterly'")

    carried_data = data.resample('M').ffill().bfill()
    carried_data = carried_data.groupby([carried_data.index.year, (carried_data.index.month - 1) // period]).ffill().bfill()
    return carried_data

# Apply carry forward for annual and quarterly data
for code in annual_codes:
    char_column = f'characteristic_{code}'
    stock_characteristics_monthly_py[char_column] = carry_forward(stock_characteristics_monthly_py[char_column], 'annual')

for code in quarterly_codes:
    char_column = f'characteristic_{code}'
    stock_characteristics_monthly_py[char_column] = carry_forward(stock_characteristics_monthly_py[char_column], 'quarterly')

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
narratives = [f'characteristic_{code}' for code in all_codes]

# Rank normalization function
def rank_normalize(df, columns):
    df[columns] = df[columns].apply(lambda x: pd.DataFrame(quantile_transform(x.values.reshape(-1, 1),
                                                                              copy=True,
                                                                              n_quantiles=len(x)).flatten(),
                                                           columns=[x.name],
                                                           index=x.index))
    df[columns] = df[columns] * 2 - 1
    return df

# Apply rank normalization by month
stock_characteristics_monthly_py = stock_characteristics_monthly_py.groupby(stock_characteristics_monthly_py.index.month).apply(rank_normalize, columns=narratives)

# Example of checking the result
print(stock_characteristics_monthly_py.head())


In [ ]:
from sklearn.preprocessing import quantile_transform
from joblib import delayed, Parallel
from itertools import product
# Set the index of 'results' to a MultiIndex with 'date' and 'permno'
#merged_df.set_index(['date', 'permno'], inplace=True)
narratives = stock_characteristics_monthly_py.drop('ret_excess', axis=1).columns.tolist()

# narratives = results.columns.tolist()

# narratives = data.drop('returns', axis=1).columns.tolist()

stock_characteristics_monthly_py.loc[:, narratives] = (stock_characteristics_monthly_py.loc[:, narratives]
                                .groupby(level='month')
                                .apply(lambda x: pd.DataFrame(quantile_transform(x,
                                                                                 copy=True,
                                                                                 n_quantiles=x.shape[0]),
                                                              columns=narratives,
                                                              index=x.index.get_level_values('permno')))
                               .mul(2).sub(1))

stock_characteristics_monthly_py.sort_index(inplace=True)
stock_characteristics_monthly_py.index.names
stock_characteristics_monthly_py.loc[:, ['ret_excess']] = stock_characteristics_monthly_py.loc[:, ['ret_excess']].clip(lower=-1, upper=1.0)


In [ ]:
# Extract column names except 'DATE' and 'permno' and format them in a list
NARRATIVES_LIST = [col for col in merged_df1.columns if col not in ['month', 'permno','ret_excess']]

# Print the list in the desired format
print(NARRATIVES_LIST)
print(len(NARRATIVES_LIST))

**Creating the factor loading dataset**

In [ ]:
from joblib import delayed, Parallel
from itertools import product
def cal_portfolio_ret(it, df):
    d, f = it[0], it[1]
    # long portfolio, qunatile 0.0~0.1; short portfolio, qunatile 0.9~1.0
    long_portfolio = df.loc[df.date == d][['permno', f]].sort_values(by=f, ascending=False)[:df.loc[df.date == d].shape[0]//10]['permno'].to_list()
    short_portfolio = df.loc[df.date == d][['permno', f]].sort_values(by=f, ascending=False)[-df.loc[df.date == d].shape[0]//10:]['permno'].to_list()
    # long-short portfolio return
    long_ret = ret_excess_df_temp.loc[ret_excess_df_temp.date == d].drop_duplicates('permno').set_index('permno').reindex(long_portfolio)['ret_excess'].dropna().mean()
    short_ret = ret_excess_df_temp.loc[ret_excess_df_temp.date == d].drop_duplicates('permno').set_index('permno').reindex(short_portfolio)['ret_excess'].dropna().mean()
    chara_ret = 0.5*(long_ret - short_ret)

    return chara_ret


##TODO: calculate portfolio returns (or download preprocessed data)
iter_list = list(product(processed_narratives.date.drop_duplicates(), NARRATIVES_LIST))
portfolio_rets = Parallel(n_jobs=-1)(delayed(cal_portfolio_ret)(it, df=processed_narratives) for it in tqdm(iter_list, colour='green', desc='Calculating Portfolio Returns'))
portfolio_rets = pd.DataFrame(np.array(portfolio_rets).reshape(-1, 180), index=processed_narratives.date.drop_duplicates(), columns=NARRATIVES_LIST).reset_index()
portfolio_rets[NARRATIVES_LIST] = portfolio_rets[NARRATIVES_LIST].astype(np.float16)


In [ ]:
import numpy as np
import pandas as pd

x_t_all_months = pd.DataFrame()
unique_months = stock_characteristics_monthly_py['month'].unique()
lambda_reg = 0.0001  # Regularization parameter

for current_month in unique_months:
    current_month_df = stock_characteristics_monthly_py[stock_characteristics_monthly_py['month'] == current_month]
    Z_t_minus_1 = current_month_df.iloc[:, 2:-1].values
    r_t = current_month_df['ret_excess'].values

    # Regularization added to Z_t_minus_1.T @ Z_t_minus_1
    regularized_matrix = Z_t_minus_1.T @ Z_t_minus_1 + lambda_reg * np.eye(Z_t_minus_1.shape[1])

    # Calculate x_t using the regularized matrix
    x_t = np.linalg.inv(regularized_matrix) @ Z_t_minus_1.T @ r_t

    x_t_df_row = pd.DataFrame(x_t.reshape(1, -1), columns=stock_characteristics_monthly_py.columns[2:-1])
    x_t_df_row['month'] = current_month
    x_t_all_months = pd.concat([x_t_all_months, x_t_df_row], ignore_index=True)

x_t_all_months.reset_index(drop=True, inplace=True)

# Convert 'month' column to datetime format
x_t_all_months['month'] = pd.to_datetime(x_t_all_months['month'])

# Set the 'month' column as the index of the DataFrame
x_t_all_months.set_index('month', inplace=True)

# Now, print the index to confirm it's a DatetimeIndex
print(x_t_all_months.index)



In [ ]:
print(x_t_all_months.head())
print(x_t_all_months.tail())

**Checking the size of the factor loading dataset and the stock characteristics dataset**

In [ ]:
from google.colab import drive
import pandas as pd
_
# Mount Google Drive
#drive.mount('/content/drive')

# Ensure you have the required library
!pip install pyarrow

# Assuming x_t_all_months and stock_characteristics_monthly_py are your dataframes
x_t_all_months.to_parquet('/content/drive/My Drive/Tidy Finance ML dataset/x_t_all_months.parquet', engine='pyarrow')
stock_characteristics_monthly_py.to_parquet('/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet', engine='pyarrow')


In [ ]:
# Step 1: Install pyarrow if not already installed
!pip install pyarrow

import pandas as pd
import os

# Assuming x_t_all_months and stock_characteristics_monthly_py are your DataFrames
# Replace these lines with your actual DataFrame creation/loading code
# x_t_all_months = pd.DataFrame(...)
# stock_characteristics_monthly_py = pd.DataFrame(...)

# Save the DataFrames as Parquet files
x_t_all_months.to_parquet('x_t_all_months.parquet', engine='pyarrow')
stock_characteristics_monthly_py.to_parquet('stock_characteristics_monthly_py.parquet', engine='pyarrow')

# Get the file sizes
x_t_all_months_size = os.path.getsize('x_t_all_months.parquet')
stock_characteristics_monthly_py_size = os.path.getsize('stock_characteristics_monthly_py.parquet')

# Print the file sizes
print(f"Size of x_t_all_months parquet file: {x_t_all_months_size} bytes")
print(f"Size of stock_characteristics_monthly_py parquet file: {stock_characteristics_monthly_py_size} bytes")
